In [48]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

from src.prepare_dataset import get_dev_df  

pd.set_option('display.max_colwidth', 1000)

In [49]:
ROBERTA_BASE_PATH = Path('/home/nik/ImperialWork/nlp/coursework/outputs/2026-03-01/16-50-48/out/ensemble/dev.txt')         
BEST_MODEL_PATH = Path('/home/nik/ImperialWork/nlp/coursework/multirun/2026-03-02/15-08-30/18/out/ensemble/dev.txt') 

ROBERTA_BASE = 'Roberta-Base Baseline'
BEST_MODEL = 'Best Model Ensemble'

In [50]:
dev_df = get_dev_df()  
y_true = dev_df['label'].astype(int).to_numpy()
texts = dev_df['text'].astype(str).to_list()

In [51]:
def read_dev_txt(path: Path) -> np.ndarray:
    lines = [ln.strip() for ln in path.read_text(encoding='utf-8').splitlines() if ln.strip() != '']
    return np.array([int(x) for x in lines], dtype=np.int64)

pred_a = read_dev_txt(ROBERTA_BASE_PATH)
pred_b = read_dev_txt(BEST_MODEL_PATH)

assert len(pred_a) == len(y_true), f'{ROBERTA_BASE} length {len(pred_a)} != dev size {len(y_true)}'
assert len(pred_b) == len(y_true), f'{BEST_MODEL} length {len(pred_b)} != dev size {len(y_true)}'

In [52]:
cmp = pd.DataFrame({
    'text': texts,
    'y_true': y_true,
    'pred_a': pred_a,
    'pred_b': pred_b,
})

cmp['a_correct'] = (cmp['pred_a'] == cmp['y_true'])
cmp['b_correct'] = (cmp['pred_b'] == cmp['y_true'])

# 4-way outcome bucket
cmp['bucket4'] = np.select([
        cmp['a_correct'] & cmp['b_correct'],
        (~cmp['a_correct']) & (~cmp['b_correct']),
        cmp['a_correct'] & (~cmp['b_correct']),
        (~cmp['a_correct']) & cmp['b_correct'],
    ],
    [
        'both_correct',
        'both_wrong',
        'A_correct_B_wrong',
        'A_wrong_B_correct',
    ],
    default='(unexpected)',
)

cmp['error_type_a'] = np.select(
    [(cmp['y_true'] == 0) & (cmp['pred_a'] == 1), (cmp['y_true'] == 1) & (cmp['pred_a'] == 0)],
    ['FP', 'FN'],
    default='OK',
)
cmp['error_type_b'] = np.select(
    [(cmp['y_true'] == 0) & (cmp['pred_b'] == 1), (cmp['y_true'] == 1) & (cmp['pred_b'] == 0)],
    ['FP', 'FN'],
    default='OK',
)

cmp['label_name'] = np.where(cmp['y_true'] == 1, 'PCL', 'nonPCL')
cmp['bucket8'] = cmp['bucket4'].astype(str) + '_' + cmp['label_name'].astype(str)

BUCKET4 = ['both_correct', 'both_wrong', 'A_correct_B_wrong', 'A_wrong_B_correct']
BUCKET8 = [f'{b}_nonPCL' for b in BUCKET4] + [f'{b}_PCL' for b in BUCKET4]

In [53]:
def sample_examples(
    df: pd.DataFrame,
    bucket8: str,
    n: int = 8,
    seed: int = 0,
    error_type_a: str | None = None,
    error_type_b: str | None = None,
) -> pd.DataFrame:
    sub = df[df['bucket8'] == bucket8]

    if error_type_a is not None:
        sub = sub[sub['error_type_a'] == error_type_a]
    if error_type_b is not None:
        sub = sub[sub['error_type_b'] == error_type_b]

    if len(sub) == 0:
        return sub

    sub = sub.sample(n=min(n, len(sub)), random_state=seed)
    cols = ['y_true', 'pred_a', 'pred_b', 'bucket4', 'bucket8', 'error_type_a', 'error_type_b', 'text']
    return sub[cols].reset_index(drop=True)


In [58]:
for b in BUCKET8:
    print(b)
    display(sample_examples(cmp, b, n=10, seed=0))

both_correct_nonPCL


,y_true,pred_a,pred_b,bucket4,bucket8,error_type_a,error_type_b,text
0,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"The deaths were due to wall collapses , drowning and electrocution , while thousands of homeless were lodged in government-run relief camps ."
1,0,0,0,both_correct,both_correct_nonPCL,OK,OK,Syrian refugee children .
2,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"Expert in Escaping North Korea Has a Message for Donald Trump <h> North Korean refugees Grace Jo ( left ) and Jung Gwang Il ( right ) appear at a news conference before attending a meeting of the United Nations Security Council at UN headquarters in New York on December 10 , 2015 . ( File Photo/REUTERS )"
3,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"Akande said : "" We are using a Community-Based Targeting template of the World Bank and as we have explained , this is the mode of identifying the poorest of the poor and the most vulnerable . """
4,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"Among those who coded in Ruby , women coders scored an average of 93 points while male coders scored 60 points . Python emerged as another favorite scripting language for women coders where they edged past male coders by 17 points ."
5,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"While China can be confident of at least a 4-1 victory over India , Indonesia has to marshal all of its resources . If India were to record wins by Saina and Parupalli Kashyap , this could put a lot of pressure on Indonesia 's doubles pairs and either Jwala Gutta or Ashwini Ponnappa could rise to the occasion in either mixed or women 's doubles , even in one of their new partnerships ."
6,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"Dr Graeme Codrington , expert on the future world of work and the disruptive forces that are shaping it argued that leadership development and executive training was in need of an overhaul . "" While we attempt to teach agility , adaptability and how to engage with the unknown , programmes are designed and controlled down to the smallest detail , leaving no room for exploration and discovery . """
7,0,0,0,both_correct,both_correct_nonPCL,OK,OK,""" The best thing that happened , 3 million people came out of the shadows , "" says Simpson . "" We called it legalization . There was no amnesty . "" It was too close to the Vietnam War . President Carter had given amnesty to young men who had fled to Canada to avoid Vietnam , and it was a flash word . Still , both critics and advocates remember the ' 86 bill as amnesty for 3 million illegal immigrants . Today 's reformers are fighting the same semantic battle , insisting that legality and an earned path to citizenship are not amnesty ."
8,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"This year the Shanghai resort plans to recruit 2,000 full-time and part-time employees , with vacancies tailored to the disabled population , said Lara Tiam , vice-president of human resources at the resort ."
9,0,0,0,both_correct,both_correct_nonPCL,OK,OK,""" These lengthy , inefficient and burdensome processes result in more hindrance than assistance to SMEs that are more in need of government assistance than larger firms to enter international markets , "" the study highlighted ."


both_wrong_nonPCL


,y_true,pred_a,pred_b,bucket4,bucket8,error_type_a,error_type_b,text
0,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,""" Your personal leadership has been critical to addressing the plight of the Rohingya who fled to safety in your country . I thank you for all you have done to assist these men , women and children in need , "" he wrote in the message ."
1,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"The past year has not been easy -- neither for the refugees , nor for those assisting them . A process complicated by the knowledge that there is no end in sight to the suffering of the most persecuted community in the world , with diplomatic deals regarding repatriation , having failed time and time again in the past ."
2,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"The majority of these people now reside , in refugee camps that lack sufficient humanitarian aid , offering refugees little more than a hopeless situation . International charitable organisations fear working in these camps as harassment and violence against those aiding Rohingya Muslims are common ."
3,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"Christmas is celebration of the birth of not merely a child , but a child who changed the destiny of humans forever . It is celebration of the fact that God wanted to be a part of the human race and so he took on flesh and blood and became human like us . We can also show unconditional love through our good deeds and helping those who are in need of our help and care . Be human and merciful ."
4,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"6 years ago she lost her husband -- he died from heart disease , and since then Wood lives alone with her children and a dog . When she learnt about Michael and Cory 's terrible life problems , she realised she had to help . This woman has a big kind heart . There were 2 empty rooms in her house , so she invited the homeless to live there . Many people might consider her action as a madness but Mel says that since her husband 's death , she does n't fear anything ."
5,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,Geeta and the other women in their Self Help Group are also proud of the fact that other women are also getting inspired by their progress story . More and more women are getting associated with Bihan with an aim to bring about positive changes in their lives and most of them are inspired by other women who have already shown the example .
6,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"The University of Technology , Jamaica through its USAID-funded Fi Wi Jamaica Project on international Human Rights Day , Thursday , December 10 , 2015 honoured ten ( 10 ) ? unsung heroes ? and one ( 1 ) institution for their transformative and enduring humanitarian work in influencing the way that Jamaican society values the most vulnerable . The awards were presented at the 2nd Annual Essence of Humanity Ubuntu Awards ceremony held at the PCJ auditorium , New Kingston ."
7,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"In an act of defiance against Hungarian authorities , which had suspended trains to Western Europe , between 1,200 and 2,000 refugees decided to walk from Keleti Station in Budapest to Vienna on Friday . Men , women , children -- even people on crutches -- marched from the late morning to evening , clutching bags filled with all the possessions they had ."
8,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"She added : "" I am really happy with the awards , but I just want to help those in need . """
9,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"In his condolence message , he said , "" All our sympathies are with the poor families as we are standing in cohesion with the Egyptian government and the people in this hour of grief . """


A_correct_B_wrong_nonPCL


...

,y_true,pred_a,pred_b,bucket4,bucket8,error_type_a,error_type_b,text
0,1,0,0,both_wrong,both_wrong_PCL,FN,FN,""" Boarding schools should be for those coming from poor families who when admitted to boarding secondary schools would be assured of their meals provided by schools , "" she said ."
1,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"After a tragic event during his previous life as an FBI agent , Sawyer lost half a leg and developed an aversion to guns . So instead of the usual muscle-bound , wise-cracking , gun-toting cliche he 's often played , here he 's a vulnerable , disabled family man caught up in a twisted extortion plot ."
2,1,0,0,both_wrong,both_wrong_PCL,FN,FN,Hollywood star Leo Di Caprio urges help for reuniting immigrant children with their families
3,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"However , this success story is not uncommon . It often happens that people from poor families have more perseverance to fight tooth an nail in business than children of rich parents who are used to get everything they want with ease . People without a strong spirit will quickly break down and drop out from the competition ."
4,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"The vast majority of girls and women caught in the exploitative global sex trade are not victims of kidnapping , like the Nigerian 276 abducted by Boko Haram , but rather of poverty . Human traffickers prey on poor families who do n't have access to education and are n't aware of their basic rights . Mired in grinding poverty , parents desperately take out loans on conditions they do n't understand , pledging their children on their debts ."
5,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"NEW DELHI : Activists and medical professionals have lauded the government 's move to increase the ambit of the disabled list to offer benefits to acid attack survivors , those suffering from chronic neurological conditions , and haemophilia and sickle cell anaemia patients . This , they said , will help in integrating them into society ."
6,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"KOLKATA : Sourav Kaibartya , a fisherman 's son who scored 94.2% in his higher secondary examination this year got entry into NIT Durgapur for engineering course . The boy was at a loss as to who will fund his education . That is when a corporate house came into his rescue . Thirty-seven students like him from West Bengal meritorious but from poor families will now get to continue their studies with an initiative from Magma Fincorp Ltd ."
7,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"But if the Supreme Court gives a favorable decision for the president , his immigration program would immediately take effect , changing the lives of eligible Filipino families and other immigrants ."
8,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"We are alarmed to learn of your recently circulated proposals that would eviscerate the Lifeline program and leave many of the most vulnerable people in the country without access to affordable communications . As you are well aware , the Lifeline program provides a modest monthly subsidy of $9.25 to connect low-income Americans to phone and internet services . As broadband prices continue to soar , and affordability continues to suffer , adoption gaps remain . The Lifeline program has proven critical for poor families and people of color who are caught on the wrong side of the digital divide ."
9,1,0,0,both_wrong,both_wrong_PCL,FN,FN,If only we had more stories that championed the brilliance of migrant workers perhaps we 'd be able to challenge the silence that permits them to be treated in such a disdainful way .


A_correct_B_wrong_PCL


,y_true,pred_a,pred_b,bucket4,bucket8,error_type_a,error_type_b,text
0,1,1,0,A_correct_B_wrong,A_correct_B_wrong_PCL,OK,FN,""" We believe in the ability of young women to achieve great things , both for themselves and for Tanzania , "" he added ."
1,1,1,0,A_correct_B_wrong,A_correct_B_wrong_PCL,OK,FN,"Hundreds of thousand Africans are graduating per year . Different from 1980s and early 1990s when college outpours got immediately absorbed in the labour market , many today are jobless and hopeless ."
2,1,1,0,A_correct_B_wrong,A_correct_B_wrong_PCL,OK,FN,"A few of Hong Kong 's mega rich , have also made a more concerted effort to improve the quality of life of those most in need , however the philanthropic approach adopted by the vast majority of them has its limitations . Philanthropy has often created a culture of dependency and has failed to tackle the root causes of social problems . For many organizations , philanthropic investment is viewed as a cost rather than an opportunity ."
3,1,1,0,A_correct_B_wrong,A_correct_B_wrong_PCL,OK,FN,"President Muhammadu Buhari is on a divine mission to rescue the Nigeria from the brink of collapse and hopelessness , Speaker of the House of Representatives , Rt . Hon. Yakubu Dogara has said ."
4,1,1,0,A_correct_B_wrong,A_correct_B_wrong_PCL,OK,FN,""" We know Uber partners with an extensive network of drivers , and Plunket nurses support some of our most vulnerable families . Working together , we can make it simple for people to help out and make a difference , "" Jarvie says ."
5,1,1,0,A_correct_B_wrong,A_correct_B_wrong_PCL,OK,FN,"She added that her office has worked relentlessly with the present administration to translate policies into concrete actions for the women , children , the aged , orphans , the disabled , and other vulnerable members of the society . Hajiya Yuguda challenged Nigerians to believe in the immense capacity and capability of the women folk in nation building in order to bring about the much needed positive change the country is yearning for ."
6,1,1,0,A_correct_B_wrong,A_correct_B_wrong_PCL,OK,FN,"Old BC : As usual , Old GB has it assy-versy . I think there should be more focus on preventing homelessness than curing it . Are there systems for actually going out , picking up the homeless , and placing them where they belong ? Homeless people are n't voluntarily walking into these shelters ."


A_wrong_B_correct_PCL


,y_true,pred_a,pred_b,bucket4,bucket8,error_type_a,error_type_b,text
0,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,""" I have a lot of sympathy for folks who are in need in the city , "" Mr Edmonds-Waters said . "" This has become an extremely expensive city to live in . The divide between those who have and those who do n't is ridiculously ginormous . """
1,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,""" It 's the largest humanitarian tragedy of our time , "" Ninette Kelley , the UN high commissioner for refugees ' representative to Lebanon , told the Guardian earlier this month in an interview . "" We do not want the world to forget that people are suffering here . """
2,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"After the satisfying evening , Manny Osme ? a left us pondering on his quote , "" Money is pressure until it is used to help people in need . """
3,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"Rather sad . Good set of pictures that tells a tale of survival , subsistence living , and hopeless nature of life in the tribal societies . Exploiting an unexpected geo-political bonanza is a temporary relief that is not sustainable . Education and economic development seem miles away ."
4,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"Yes , DMC you are very good at adding numbers , I am sure the dead and homeless are really thankful that you have the numbers . But as a government department I do not see any action in preventing , educating , keeping sos supply depots , or indeed any action , apart from the armed forces providing aid and the common man ."
5,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,""" In addition , the Members of Parliament will visit two projects that focus on the inclusion of street children and victims of acid attacks to demonstrate their solidarity with those vulnerable groups in need of support . Finally , on April 29 the delegation will brief the journalists on the outcome of the visit at a press conference . """
6,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"From mother ( played by Bollywood actress Seema Biswas of Bandit Queen fame ) who rejected him and a society that offered no respite , Siva was , in a nutshell , a hopeless street vagabond ."
7,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"In his final year as president , Mr S R Nathan - together with a few of his close friends - started discussing with me the idea of starting a philanthropic fund to help "" uplift "" children from poor families ."
8,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,McAfee said he was helping the locals by feeding poor families and providing them with jobs .
9,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"In Dublin , the Church of Ireland Archbishop Dr Michael Jackson reflected on the year drawing to a close and recalled the "" horrific conflagration at the halting site in Carrickmines "" and the death of Garda Tony Golden in Omeath , as well as the "" refugees fleeing from Syria "" and other parts of the Middle East and the "" forgotten peoples of Africa "" ."


In [56]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


# -----------------------
# Stats helpers
# -----------------------
def binary_metrics(y: np.ndarray, p: np.ndarray) -> dict:
    acc = float(accuracy_score(y, p))
    prec, rec, f1, _ = precision_recall_fscore_support(y, p, average='binary', zero_division=0)
    return {
        'accuracy': acc,
        'precision_pos': float(prec),
        'recall_pos': float(rec),
        'f1_pos': float(f1),
    }

def describe_confusion(y: np.ndarray, p: np.ndarray) -> dict:
    cm = confusion_matrix(y, p, labels=[0, 1])
    tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])
    return {'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}

def bucket_counts(df: pd.DataFrame, bucket_col: str) -> pd.DataFrame:
    out = df[bucket_col].value_counts().rename_axis(bucket_col).reset_index(name='count')
    out['pct'] = out['count'] / len(df)
    return out

def label_breakdown(df: pd.DataFrame) -> pd.DataFrame:
    return pd.crosstab(df['bucket4'], df['y_true'], normalize='index').rename(columns={0: 'true=0', 1: 'true=1'})


# -----------------------
# Compute stats
# -----------------------
y = cmp['y_true'].to_numpy()

stats_overall = pd.DataFrame(
    [
        {'model': 'A', **binary_metrics(y, cmp['pred_a'].to_numpy()), **describe_confusion(y, cmp['pred_a'].to_numpy())},
        {'model': 'B', **binary_metrics(y, cmp['pred_b'].to_numpy()), **describe_confusion(y, cmp['pred_b'].to_numpy())},
    ]
)

disagree_rate = float(np.mean(cmp['pred_a'] != cmp['pred_b']))
a_better = int((cmp['a_correct'] & ~cmp['b_correct']).sum())
b_better = int((~cmp['a_correct'] & cmp['b_correct']).sum())

stats_compare = pd.DataFrame([{
    'disagreement_rate': disagree_rate,
    'A_correct_B_wrong_count': a_better,
    'A_wrong_B_correct_count': b_better,
    'net_A_minus_B': a_better - b_better,
}])

bucket4_counts = bucket_counts(cmp, 'bucket4')
bucket8_counts = bucket_counts(cmp, 'bucket8')
bucket4_label_mix = label_breakdown(cmp)

fp_fn_by_model = pd.DataFrame({
    'A': cmp['error_type_a'].value_counts(),
    'B': cmp['error_type_b'].value_counts(),
}).fillna(0).astype(int)

# -----------------------
# Display
# -----------------------
print('=== Overall metrics ===')
display(stats_overall)

print('=== Head-to-head comparison ===')
display(stats_compare)

print('=== Bucket4 counts ===')
display(bucket4_counts)

print('=== Bucket4 label mix (row-normalised) ===')
display(bucket4_label_mix)

print('=== Bucket8 counts (8 buckets) ===')
display(bucket8_counts)

print('=== Error type counts (FP/FN/OK) by model ===')
display(fp_fn_by_model)


=== Overall metrics ===


,model,accuracy,precision_pos,recall_pos,f1_pos,TN,FP,FN,TP
0,A,0.927889,0.655844,0.507538,0.572238,1842,53,98,101
1,B,0.939351,0.719512,0.592965,0.650138,1849,46,81,118


=== Head-to-head comparison ===


,disagreement_rate,A_correct_B_wrong_count,A_wrong_B_correct_count,net_A_minus_B
0,0.037249,27,51,-24


=== Bucket4 counts ===


...

y_true,true=0,true=1
bucket4,,
A_correct_B_wrong,0.740741,0.259259
A_wrong_B_correct,0.529412,0.470588
both_correct,0.950939,0.049061
both_wrong,0.260000,0.740000


=== Bucket8 counts (8 buckets) ===


,bucket8,count,pct
0,both_correct_nonPCL,1822,0.870105
1,both_correct_PCL,94,0.044890
2,both_wrong_PCL,74,0.035339
3,A_wrong_B_correct_nonPCL,27,0.012894
4,both_wrong_nonPCL,26,0.012416
5,A_wrong_B_correct_PCL,24,0.011461
6,A_correct_B_wrong_nonPCL,20,0.009551
7,A_correct_B_wrong_PCL,7,0.003343


=== Error type counts (FP/FN/OK) by model ===


,A,B
OK,1943,1967
FN,98,81
FP,53,46


# BestModel

This folder points to the implementation of the best-performing model described in the report. The core code lives in `src/`; this README links to the exact components.

Weights are provided via Google Drive: https://drive.google.com/drive/folders/1md0LZDrtph17YcO5gDjTFDpL4Un3j-HX?usp=sharing. Predictions in `dev.txt` and `test.txt` at the repo root.

---

## Main components

- **Model definition (RoBERTa + heads):** 
  https://github.com/NikWill2003/DPM_binary_classification/blob/main/src/model.py#L11-L72

- **Training loop + loss composition:** 
  https://github.com/NikWill2003/DPM_binary_classification/blob/main/src/trainer.py#L28-L419

- **Dataset preparation / split ordering:**
  https://github.com/NikWill2003/DPM_binary_classification/blob/main/src/prepare_dataset.py#L133-L213

- **Metrics + writing predictions:**  
  https://github.com/NikWill2003/DPM_binary_classification/blob/main/src/utils.py#L114-L178

---

## Additions described in the report

- **Encoder upgrade (RoBERTa-large):** 
  https://github.com/NikWill2003/DPM_binary_classification/blob/main/src/model.py#L22-L22

- **CORAL ordinal objective:** 
  label construction: https://github.com/NikWill2003/DPM_binary_classification/blob/main/src/prepare_dataset.py#L103-L130

- **Class-weighted loss:**  
  https://github.com/NikWill2003/DPM_binary_classification/blob/main/src/trainer.py#L171-L185

- **Focal scaling:** 
  https://github.com/NikWill2003/DPM_binary_classification/blob/main/src/trainer.py#L186-L220

- **Auxiliary multi-label PCL-type head:**
  loss definition: https://github.com/NikWill2003/DPM_binary_classification/blob/main/src/trainer.py#L212-L217
  trainer usage: https://github.com/NikWill2003/DPM_binary_classification/blob/main/src/trainer.py#L289-L324

- **LLRD:** parameter grouping functions  
  https://github.com/NikWill2003/DPM_binary_classification/blob/main/src/param_groups.py#L21-L67

- **Cross-fold ensembling:**  
  https://github.com/NikWill2003/DPM_binary_classification/blob/main/train.py#L135-L166

---

## Best run command

```
python train.py \
  seed=0 \
  cv.enabled=true \
  cv.n_splits=4 \
  cv.shuffle=true \
  cv.ensemble=true \
  train.save_best = true \
  train.mixed_precision=bf16 \
  train.max_steps=2000 \
  train.eval_every_steps=100 \
  train.early_stopping_evals=4 \
  optim.optim=adamw \
  optim.wd=0.01 \
  optim.warmup_ratio=0.05 \
  optim.lldr.enabled=true \
  optim.lldr.lr_head=1e-5 \
  optim.lldr.lr_top=1e-5 \
  optim.lldr.layer_decay=0.95 \
  loss.use_coral_objective=true \
  loss.use_weighted_loss=true \
  loss.focal_loss_lambda=1.0 \
  loss.aux_multilabel_plc_loss_weight=0.5 \
  loss.use_weighted_aux_loss=false \
  model.encoder_name=roberta-large \
  model.head_dropout=0.1 \
  model.use_linear_head=true
```